In [ ]:
import csv
import time
import logging
import re
from pathlib import Path
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
import undetected_chromedriver as uc
from selenium.webdriver.support.ui import WebDriverWait

START_URL = 'https://www.livinginsider.com/living_zone/45/all/all/1/%E0%B9%80%E0%B8%8A%E0%B8%B5%E0%B8%A2%E0%B8%87%E0%B9%83%E0%B8%AB%E0%B8%A1%E0%B9%88.html'
OUTPUT_CSV_FILE = Path('livinginsider_listing_urls.csv')
PAGE_TIMEOUT = 45
MAX_PAGES = 20

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def build_page_url(url: str, page: int) -> str:
    p = urlparse(url)
    parts = [x for x in p.path.split('/') if x]
    k = next((i for i, seg in enumerate(parts) if re.fullmatch(r'\d+', seg)), None)
    
    if k is None:
        new_path = p.path.rstrip('/') if page <= 1 else f"{p.path.rstrip('/')}/{page}"
    else:
        parts[k] = '1' if page <= 1 else str(page)
        new_path = '/' + '/'.join(parts)
        
    return urlunparse((p.scheme, p.netloc, new_path, p.params, urlencode(dict(parse_qsl(p.query)), doseq=True), p.fragment))

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    options.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    })
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    return uc.Chrome(options=options, version_main=145)

def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    js = """
    return [...new Set(
        [...document.querySelectorAll("a[href*='/livingdetail/'][href$='.html'], a[href^='/livingdetail/'][href$='.html']")]
        .map(a => a.getAttribute('href'))
        .filter(h => h && !h.includes('bclick') && !h.includes('stories') && !h.includes('banner'))
    )];
    """
    return [f"{base_url}{h}" if h.startswith("/") else h for h in driver.execute_script(js)]

def scroll_page(driver: uc.Chrome, rounds: int, pause: float):
    last_h = 0
    for _ in range(rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h

def is_ready(driver: uc.Chrome) -> bool:
    return driver.execute_script("return document.readyState") == "complete"

def main():
    driver = setup_driver()
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
    
    all_urls = set()
    page = 1
    last_first = ""

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(build_page_url(START_URL, page))
        WebDriverWait(driver, PAGE_TIMEOUT).until(is_ready)

        while page <= MAX_PAGES:
            logging.info(f"Processing Page {page}...")
            scroll_page(driver, rounds=20, pause=0.8)
            urls_now = extract_links(driver, base_url)

            if not urls_now or (page > 1 and urls_now[0] == last_first):
                break

            last_first = urls_now[0]
            all_urls.update(urls_now)
            logging.info(f"Found {len(urls_now)} listings (Total unique: {len(all_urls)})")

            page += 1
            if page > MAX_PAGES:
                break

            prev_last = urls_now[-1]
            driver.get(build_page_url(START_URL, page))
            
            t0 = time.time()
            changed = False
            
            while time.time() - t0 < PAGE_TIMEOUT:
                try:
                    WebDriverWait(driver, 3).until(is_ready)
                except Exception:
                    pass
                    
                scroll_page(driver, rounds=6, pause=0.6)
                cur = extract_links(driver, base_url)
                
                if cur and (cur[0] != last_first or cur[-1] != prev_last):
                    changed = True
                    break
                time.sleep(0.3)

            if not changed:
                break

    except KeyboardInterrupt:
        logging.info("Interrupted by user.")
    except Exception as e:
        logging.error(f"Error: {e}")
    finally:
        driver.quit()

    if all_urls:
        with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['ListingURL'])
            writer.writerows([[u] for u in sorted(all_urls)])
        logging.info(f"Saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
import csv
import logging
import re
import sys
import time
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/Scraping')
INPUT_CSV = BASE_DIR / 'livinginsider_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'livinginsider_scraped_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    
    return uc.Chrome(options=options, version_main=145)

def get_text_safe(driver: uc.Chrome, by: By, selector: str) -> str:
    try:
        return driver.find_element(by, selector).text.strip()
    except NoSuchElementException:
        return ""

def parse_coords(text: str) -> str:
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)", text)
    return f"{m.group(1)},{m.group(2)}" if m else ""

def click_consent(driver: uc.Chrome):
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, "#onetrust-accept-btn-handler") + \
               driver.find_elements(By.XPATH, "//*[self::a or self::button][contains(., 'ยอมรับ')]")
        
        for btn in btns:
            if btn.is_displayed() and btn.is_enabled():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.3)
                break
    except Exception:
        pass

def scrape_one(driver: uc.Chrome, wait: WebDriverWait, url: str) -> dict:
    retries = 3
    for attempt in range(retries):
        try:
            driver.get(url)
            click_consent(driver)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.font_sarabun.show-title")))
            
            description = get_text_safe(driver, By.CSS_SELECTOR, "#desc-text-nl .wordwrap-box .wordwrap")
            
            vc = driver.find_elements(By.CSS_SELECTOR, ".box-show-view-click .text-custom-gray-new")
            views = vc[0].text.strip() if len(vc) > 0 else ""
            clicks = vc[1].text.strip() if len(vc) > 1 else ""

            phone_nodes = driver.find_elements(By.CSS_SELECTOR, "a.p-phone-contact[data-zcgrbcb]")
            phones_masked = ", ".join(sorted(set(n.get_attribute("data-zcgrbcb") for n in phone_nodes if n.get_attribute("data-zcgrbcb"))))

            return {
                "URL": url,
                "Title": get_text_safe(driver, By.CSS_SELECTOR, "h1.font_sarabun.show-title"),
                "Price": get_text_safe(driver, By.CSS_SELECTOR, ".show_price_topic .price_topic b"),
                "PricePerArea": get_text_safe(driver, By.CSS_SELECTOR, ".show_price_topic .price_cal_area_text"),
                "PropertyType": get_text_safe(driver, By.CSS_SELECTOR, ".box_tag_topic_detail .box_tag_building"),
                "PostType": get_text_safe(driver, By.CSS_SELECTOR, ".box_tag_topic_detail .box_tag_posttype"),
                "Size": get_text_safe(driver, By.CSS_SELECTOR, ".detail_property_list_des_new .detail-property-list-text"),
                "Description": description,
                "CreatedDate": get_text_safe(driver, By.CSS_SELECTOR, ".row-detail-time .font_10_date"),
                "Boosted": get_text_safe(driver, By.XPATH, "//div[contains(@class,'row-detail-time')]//span[contains(text(),'ดันประกาศล่าสุด')]"),
                "Location": get_text_safe(driver, By.CSS_SELECTOR, ".form-group.group-location-detail .detail-text-zone a"),
                "Views": views,
                "Clicks": clicks,
                "Coordinates": parse_coords(description),
                "MaskedContacts": phones_masked
            }
            
        except TimeoutException:
            if attempt == retries - 1:
                logging.warning(f"Timeout on {url} after {retries} attempts.")
                return None
            time.sleep(1)
        except Exception as e:
             if attempt == retries - 1:
                logging.error(f"Error scraping {url}: {e}")
                return None
             time.sleep(1)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.error(f"Input file not found: {INPUT_CSV}")
        sys.exit(1)

    with INPUT_CSV.open("r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    logging.info(f"Starting to scrape {len(urls)} items...")
    driver = setup_driver()
    wait = WebDriverWait(driver, 8)
    
    rows = []

    try:
        with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
            fieldnames = [
                "URL", "Title", "Price", "PricePerArea", "PropertyType", "PostType",
                "Size", "Description", "CreatedDate", "Boosted", "Location",
                "Views", "Clicks", "Coordinates", "MaskedContacts"
            ]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for i, url in enumerate(urls, 1):
                result = scrape_one(driver, wait, url)
                if result:
                    writer.writerow(result)
                    rows.append(result)
                
                if i % 10 == 0:
                    logging.info(f"Progress: [{i}/{len(urls)}] scraped.")
                
    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()